# Lab 02 — archived notebook

This notebook preserves the uploaded calculation code and cached console results. Computer-specific paths were replaced with relative paths. Its 13 original TXT inputs are **not included** in the upload; the four supplied XLSX files give different fits. These cached results are historical, not a fresh run.

Use [`analysis.ipynb`](analysis.ipynb) or [`../src/analyze.py`](../src/analyze.py) for the runnable analysis of the supplied Excel profiles. See [provenance](../PROVENANCE.md) and [analysis review](../ANALYSIS_REVIEW.md). Original code authorship/AI assistance was not specified in the supplied Lab 02 files; the portfolio refactor and documentation were prepared with AI assistance.


In [1]:

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from docx import Document
from docx.shared import Inches

# -------------------------------------------------------
# 1) USER SETTINGS
# -------------------------------------------------------
# Folder where 25.txt, 50.txt, ..., 200.txt live
data_dir = r"../data/raw/line-profiles"   # <<< CHANGE THIS

# Where to save the Word file and figures
output_dir = data_dir  # you can change this if you want
os.makedirs(output_dir, exist_ok=True)

output_docx = os.path.join(output_dir, "Beam_Diameter_Results.docx")

# Distances (cm) and corresponding filenames
distances_cm = [25, 50, 75, 100, 125, 150, 175, 200]
filenames = [f"{d}.txt" for d in distances_cm]


# -------------------------------------------------------
# 2) GAUSSIAN MODEL + HELPERS
# -------------------------------------------------------
def gaussian_1d(x, I0, x0, w, offset):
    """
    1D Gaussian:
        I(x) = I0 * exp(-2 * (x - x0)^2 / w^2) + offset

    w is the 1/e^2 radius.
    """
    return I0 * np.exp(-2.0 * (x - x0)**2 / w**2) + offset


def load_profile(path):
    """
    Load a line profile from a text file.

    - If file has 1 column: it's y only → x = 0,1,2,...
    - If file has 2+ columns: use first column as x, second as y.
    """
    data = np.loadtxt(path)

    if data.ndim == 1:
        y = data.astype(float)
        x = np.arange(len(y), dtype=float)
    elif data.ndim == 2:
        if data.shape[1] == 1:
            y = data[:, 0].astype(float)
            x = np.arange(len(y), dtype=float)
        else:
            x = data[:, 0].astype(float)
            y = data[:, 1].astype(float)
    else:
        raise ValueError(f"Unexpected data shape in {path}: {data.shape}")

    return x, y


def fit_profile(x, y):
    """
    Fit a Gaussian to y(x). Returns best-fit params and covariance.
    """
    # Background-subtracted copy just for initial guesses
    y_bs = y - np.min(y)

    I0_guess = np.max(y_bs)
    x0_guess = x[np.argmax(y_bs)]
    w_guess = (x.max() - x.min()) / 10.0  # rough guess
    offset_guess = np.min(y)

    p0 = [I0_guess, x0_guess, w_guess, offset_guess]

    popt, pcov = curve_fit(gaussian_1d, x, y, p0=p0, maxfev=10000)
    return popt, pcov


# -------------------------------------------------------
# 3) SET UP WORD DOCUMENT
# -------------------------------------------------------
doc = Document()
doc.add_heading("Gaussian Beam Profiles and Beam Diameter vs Distance", level=1)
doc.add_paragraph(f"Data directory: {data_dir}")
doc.add_paragraph("This document contains Gaussian fits to the transverse beam "
                  "profiles at different distances from the laser, as well as "
                  "a summary plot of the beam diameter as a function of position.")


# -------------------------------------------------------
# 4) LOOP OVER ALL DISTANCES, FIT GAUSSIANS + SAVE PLOTS
# -------------------------------------------------------
beam_diameters_px = []
beam_diameter_err_px = []
fit_results = []  # (distance, diameter, diameter_err)

for d, fname in zip(distances_cm, filenames):
    path = os.path.join(data_dir, fname)
    x, y = load_profile(path)
    popt, pcov = fit_profile(x, y)

    I0, x0, w, offset = popt
    diameter = 2.0 * w  # 1/e^2 diameter in "x units" (usually pixels)

    # Uncertainty on w → uncertainty on diameter
    try:
        w_err = np.sqrt(pcov[2, 2])
        diameter_err = 2.0 * w_err
    except Exception:
        w_err = np.nan
        diameter_err = np.nan

    beam_diameters_px.append(diameter)
    beam_diameter_err_px.append(diameter_err)
    fit_results.append((d, diameter, diameter_err, w, w_err))

    # ---------- Individual fit plot for this distance ----------
    plt.figure()
    plt.plot(x, y, "k.", label="Data")

    # Smooth curve for the fit
    x_dense = np.linspace(x.min(), x.max(), 1000)
    plt.plot(x_dense, gaussian_1d(x_dense, *popt), "r-", label="Gaussian fit")

    plt.xlabel("Position (pixels or given x units)")
    plt.ylabel("Intensity (arb. units)")
    plt.title(f"Beam profile and Gaussian fit at {d} cm")
    plt.legend()
    plt.tight_layout()

    # Save figure to file
    fig_filename = os.path.join(output_dir, f"BeamProfile_{d}cm.png")
    plt.savefig(fig_filename, dpi=300)
    plt.close()

    # Add to Word doc
    doc.add_heading(f"Beam profile at {d} cm", level=2)
    p = doc.add_paragraph()
    p.add_run(
        f"Gaussian fit parameters: I0 = {I0:.2f}, x0 = {x0:.2f}, "
        f"w (radius) = {w:.2f}, offset = {offset:.2f}."
    )
    if np.isfinite(w_err):
        doc.add_paragraph(
            f"Uncertainty on w: ±{w_err:.2f} (so diameter = {diameter:.2f} ± {diameter_err:.2f} pixels)."
        )
    else:
        doc.add_paragraph(
            f"Diameter = {diameter:.2f} pixels (uncertainty could not be estimated)."
        )

    doc.add_picture(fig_filename, width=Inches(4.5))


# -------------------------------------------------------
# 5) SUMMARY PLOT: BEAM DIAMETER VS POSITION
# -------------------------------------------------------
beam_diameters_px = np.array(beam_diameters_px)
beam_diameter_err_px = np.array(beam_diameter_err_px)

plt.figure()
plt.errorbar(distances_cm, beam_diameters_px,
             yerr=beam_diameter_err_px,
             fmt="o-", capsize=4)

plt.xlabel("Distance from laser (cm)")
plt.ylabel("Beam diameter (pixels, 1/e²)")
plt.title("Gaussian beam diameter vs distance")
plt.grid(True)
plt.tight_layout()

summary_fig = os.path.join(output_dir, "BeamDiameter_vs_Distance.png")
plt.savefig(summary_fig, dpi=300)
plt.close()

# Add summary plot to Word doc
doc.add_heading("Beam diameter vs distance", level=2)
doc.add_paragraph(
    "The following plot shows the fitted 1/e² beam diameter (in pixel units) "
    "as a function of distance from the laser."
)
doc.add_picture(summary_fig, width=Inches(4.5))


# -------------------------------------------------------
# 6) ADD NUMERIC RESULTS TABLE TO WORD
# -------------------------------------------------------
doc.add_heading("Numeric results", level=2)
doc.add_paragraph(
    "Table of fitted beam diameters (1/e²) as a function of distance from the laser."
)

table = doc.add_table(rows=1, cols=4)
hdr_cells = table.rows[0].cells
hdr_cells[0].text = "Distance (cm)"
hdr_cells[1].text = "Diameter (pixels)"
hdr_cells[2].text = "σ(Diameter) (pixels)"
hdr_cells[3].text = "w (radius, pixels)"

for d, D, De, w, w_err in fit_results:
    row_cells = table.add_row().cells
    row_cells[0].text = f"{d}"
    row_cells[1].text = f"{D:.2f}"
    if np.isfinite(De):
        row_cells[2].text = f"{De:.2f}"
    else:
        row_cells[2].text = "N/A"
    if np.isfinite(w):
        if np.isfinite(w_err):
            row_cells[3].text = f"{w:.2f} ± {w_err:.2f}"
        else:
            row_cells[3].text = f"{w:.2f}"
    else:
        row_cells[3].text = "N/A"

# -------------------------------------------------------
# 7) SAVE WORD DOCUMENT
# -------------------------------------------------------
doc.save(output_docx)

print("Distance (cm)   Diameter (pixels)")
for d, D, De in zip(distances_cm, beam_diameters_px, beam_diameter_err_px):
    if np.isfinite(De):
        print(f"{d:5.0f}           {D:8.2f} ± {De:6.2f}")
    else:
        print(f"{d:5.0f}           {D:8.2f}")

print(f"\nWord document saved as:\n{output_docx}")


Distance (cm)   Diameter (pixels)
   25             379.62 ±   3.25
   50             489.50 ±   3.58
   75             470.87 ±   8.02
  100             573.27 ±   5.80
  125             917.81 ±  25.61
  150             676.28 ±   9.72
  175             840.56 ±  12.01
  200             769.96 ±   7.85

Word document saved as:
../results/archived-run/Beam_Diameter_Results.docx


In [2]:
# -*- coding: utf-8 -*-
"""
Fit Gaussian waists for three lenses (25.4mm, 100mm, 200mm)
and append all results (plots + text) to the same Word file
created by the line-profile script.
"""

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from docx import Document
from docx.shared import Inches

# -------------------------------------------------------
# 1) USER SETTINGS
# -------------------------------------------------------
# Folder where 25.4mm.txt, 100mm.txt, 200mm.txt live
data_dir = r"../data/raw/lenses"  # <<< CHANGE THIS if needed

# Path to the SAME Word document created by the first script
output_docx = r"../results/archived-run/Beam_Diameter_Results.docx"  # <<< MATCH FIRST SCRIPT

# Directory where you want to save the lens plots
# (here we put them in the same folder as the Word file)
output_dir = os.path.dirname(output_docx)
os.makedirs(output_dir, exist_ok=True)

# (focal_length_mm, filename)
lens_files = [
    (25.4, "25.4mm.txt"),
    (100.0, "100mm.txt"),
    (200.0, "200mm.txt"),
]

# If you want physical waists, set your camera pixel size (in micrometres)
# from the camera spec sheet, e.g. 5.3 µm, 4.8 µm, etc.
pixel_size_um = None   # e.g. 5.3  # <<< PUT NUMBER HERE IF KNOWN


# -------------------------------------------------------
# 2) GAUSSIAN MODEL + HELPERS
# -------------------------------------------------------
def gaussian_1d(x, I0, x0, w, offset):
    """
    1D Gaussian:
        I(x) = I0 * exp[-2 * (x - x0)^2 / w^2] + offset

    w is the 1/e^2 radius (beam waist).
    """
    return I0 * np.exp(-2.0 * (x - x0) ** 2 / w ** 2) + offset


def load_profile(path):
    """
    Load a transverse beam profile from a text file.

    - If file has 1 column: it's y only → x = 0,1,2,...
    - If file has 2+ columns: use first column as x, second as y.
    """
    data = np.loadtxt(path)

    if data.ndim == 1:
        y = data.astype(float)
        x = np.arange(len(y), dtype=float)
    elif data.ndim == 2:
        if data.shape[1] == 1:
            y = data[:, 0].astype(float)
            x = np.arange(len(y), dtype=float)
        else:
            x = data[:, 0].astype(float)
            y = data[:, 1].astype(float)
    else:
        raise ValueError(f"Unexpected data shape in {path}: {data.shape}")

    return x, y


def fit_profile(x, y):
    """
    Fit a Gaussian to y(x). Returns (best-fit params, covariance matrix).
    """
    # Use a background-subtracted copy for initial guesses
    y_bs = y - np.min(y)

    I0_guess = np.max(y_bs)
    x0_guess = x[np.argmax(y_bs)]
    w_guess = (x.max() - x.min()) / 10.0  # very rough guess of width
    offset_guess = np.min(y)

    p0 = [I0_guess, x0_guess, w_guess, offset_guess]

    popt, pcov = curve_fit(gaussian_1d, x, y, p0=p0, maxfev=10000)
    return popt, pcov


# -------------------------------------------------------
# 3) LOOP OVER LENSES, FIT + SAVE PLOTS
# -------------------------------------------------------
results = []  # will hold (f_mm, I0, x0, w_pix, w_err_pix, w_phys_um, w_phys_err_um)

for f_mm, fname in lens_files:
    path = os.path.join(data_dir, fname)
    print(f"\n--- Lens f = {f_mm} mm, file = {fname} ---")

    x, y = load_profile(path)
    popt, pcov = fit_profile(x, y)

    I0, x0, w_pix, offset = popt

    # 1σ uncertainty on w from covariance
    try:
        w_err_pix = np.sqrt(pcov[2, 2])
    except Exception:
        w_err_pix = np.nan

    # Optional: convert to physical units if pixel size known
    if pixel_size_um is not None:
        w_phys_um = w_pix * pixel_size_um
        w_phys_err_um = w_err_pix * pixel_size_um if np.isfinite(w_err_pix) else np.nan
    else:
        w_phys_um = None
        w_phys_err_um = None

    results.append((f_mm, I0, x0, w_pix, w_err_pix, offset, w_phys_um, w_phys_err_um))

    # ---------- Plot data + Gaussian fit ----------
    x_dense = np.linspace(x.min(), x.max(), 1000)
    y_fit = gaussian_1d(x_dense, *popt)

    plt.figure()
    plt.plot(x, y, "k.", label="Data")
    plt.plot(x_dense, y_fit, "r-", label="Gaussian fit")
    plt.xlabel("Position (pixel index or given x units)")
    plt.ylabel("Intensity (arb. units)")
    plt.title(f"Focused beam profile, f = {f_mm} mm")
    plt.legend()
    plt.tight_layout()

    # Save figure to file (in same folder as Word doc)
    fig_filename = os.path.join(output_dir, f"LensProfile_f{f_mm:.1f}mm.png")
    plt.savefig(fig_filename, dpi=300)
    plt.close()


# -------------------------------------------------------
# 4) PRINT SUMMARY OF WAIST SIZES (console)
# -------------------------------------------------------
print("\n==== Fitted beam waists (1/e^2 radii) ====")
header = "f (mm)   w (pixels)      σ_w (pixels)"
if pixel_size_um is not None:
    header += "      w (µm)        σ_w (µm)"
print(header)

for (f_mm, I0, x0, w_pix, w_err_pix, offset,
     w_phys_um, w_phys_err_um) in results:
    if pixel_size_um is None:
        print(f"{f_mm:6.1f}   {w_pix:10.2f}   {w_err_pix:10.2f}")
    else:
        print(
            f"{f_mm:6.1f}   {w_pix:10.2f}   {w_err_pix:10.2f}"
            f"   {w_phys_um:10.1f}   {w_phys_err_um:10.1f}"
        )


# -------------------------------------------------------
# 5) APPEND RESULTS TO EXISTING WORD DOCUMENT
# -------------------------------------------------------
# Open existing document created by first script
doc = Document(output_docx)

# Add a top-level section for lenses
doc.add_page_break()
doc.add_heading("Focused Beam Waists for Different Lenses", level=1)
doc.add_paragraph(
    "This section summarizes the Gaussian fits to the focused beam profiles "
    "obtained using lenses of different focal lengths. Each plot shows the "
    "measured transverse profile and the corresponding 1D Gaussian fit."
)

# Add one subsection per lens with its figure and parameters
for (f_mm, I0, x0, w_pix, w_err_pix, offset,
     w_phys_um, w_phys_err_um) in results:

    doc.add_heading(f"Lens with focal length f = {f_mm:.1f} mm", level=2)

    # Text with fitted parameters
    para = doc.add_paragraph()
    para.add_run(
        f"Gaussian fit parameters (in pixel units):\n"
        f"I0 = {I0:.2f}, x0 = {x0:.2f}, w = {w_pix:.2f}, "
        f"offset = {offset:.2f}.\n"
    )
    if np.isfinite(w_err_pix):
        para.add_run(
            f"Uncertainty on w: ±{w_err_pix:.2f} pixels "
            f"(1/e² radius)."
        )
    else:
        para.add_run("Uncertainty on w could not be estimated from the fit.")

    if pixel_size_um is not None and w_phys_um is not None:
        para.add_run(
            f"\nIn physical units: w = {w_phys_um:.1f} ± {w_phys_err_um:.1f} µm "
            f"(using pixel size {pixel_size_um} µm)."
        )

    # Add corresponding figure
    fig_filename = os.path.join(output_dir, f"LensProfile_f{f_mm:.1f}mm.png")
    if os.path.exists(fig_filename):
        doc.add_picture(fig_filename, width=Inches(4.5))
    else:
        doc.add_paragraph(f"(Figure file not found: {fig_filename})")


# Add a summary table
doc.add_heading("Summary of fitted waists", level=2)
doc.add_paragraph(
    "The table below lists the fitted 1/e² beam radii for each lens. "
    "If the camera pixel size is known, the waists can also be expressed "
    "in physical units."
)

if pixel_size_um is None:
    table = doc.add_table(rows=1, cols=3)
    hdr = table.rows[0].cells
    hdr[0].text = "Focal length f (mm)"
    hdr[1].text = "w (pixels)"
    hdr[2].text = "σ_w (pixels)"
else:
    table = doc.add_table(rows=1, cols=5)
    hdr = table.rows[0].cells
    hdr[0].text = "Focal length f (mm)"
    hdr[1].text = "w (pixels)"
    hdr[2].text = "σ_w (pixels)"
    hdr[3].text = "w (µm)"
    hdr[4].text = "σ_w (µm)"

for (f_mm, I0, x0, w_pix, w_err_pix, offset,
     w_phys_um, w_phys_err_um) in results:
    row = table.add_row().cells
    row[0].text = f"{f_mm:.1f}"
    row[1].text = f"{w_pix:.2f}"
    row[2].text = f"{w_err_pix:.2f}" if np.isfinite(w_err_pix) else "N/A"

    if pixel_size_um is not None:
        row[3].text = (
            f"{w_phys_um:.1f}" if w_phys_um is not None and np.isfinite(w_phys_um) else "N/A"
        )
        row[4].text = (
            f"{w_phys_err_um:.1f}"
            if w_phys_err_um is not None and np.isfinite(w_phys_err_um)
            else "N/A"
        )

# Save the updated document
doc.save(output_docx)

print(f"\nUpdated Word document saved as:\n{output_docx}")



--- Lens f = 25.4 mm, file = 25.4mm.txt ---

--- Lens f = 100.0 mm, file = 100mm.txt ---

--- Lens f = 200.0 mm, file = 200mm.txt ---

==== Fitted beam waists (1/e^2 radii) ====
f (mm)   w (pixels)      σ_w (pixels)
  25.4       117.95         0.54
 100.0        37.55         1.67
 200.0        59.07         0.83

Updated Word document saved as:
../results/archived-run/Beam_Diameter_Results.docx


In [3]:
# -*- coding: utf-8 -*-
"""
Gaussian fits for beam expander profiles and append
results (plots + text) to the same Word file used before.

Expected text files in data_dir:
    Before.txt  -> transverse profile before beam expander
    After.txt   -> transverse profile after beam expander
"""

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from docx import Document
from docx.shared import Inches

# -------------------------------------------------------
# 1) USER SETTINGS
# -------------------------------------------------------
# Folder where Before.txt and After.txt live
data_dir = r"../data/raw/expander"  # <<< CHANGE THIS if needed

before_path = os.path.join(data_dir, "Before.txt")
after_path  = os.path.join(data_dir, "After.txt")

# Path to the SAME Word document created by the first script
output_docx = r"../results/archived-run/Beam_Diameter_Results.docx"  # <<< MATCH FIRST SCRIPT

# Directory where you want to save the expander plots
output_dir = os.path.dirname(output_docx)
os.makedirs(output_dir, exist_ok=True)


# -------------------------------------------------------
# 2) GAUSSIAN MODEL + HELPERS
# -------------------------------------------------------
def gaussian_1d(x, I0, x0, w, offset):
    """
    1D Gaussian with 1/e^2 radius w:
        I(x) = I0 * exp[-2 (x - x0)^2 / w^2] + offset
    """
    return I0 * np.exp(-2.0 * (x - x0) ** 2 / w ** 2) + offset


def load_profile(path):
    """
    Load a transverse beam profile from a text file.

    If file has:
      - 1 column: y only -> x = 0, 1, 2, ...
      - 2+ columns: use first column as x and second as y.
    """
    data = np.loadtxt(path)

    if data.ndim == 1:
        y = data.astype(float)
        x = np.arange(len(y), dtype=float)
    elif data.ndim == 2:
        if data.shape[1] == 1:
            y = data[:, 0].astype(float)
            x = np.arange(len(y), dtype=float)
        else:
            x = data[:, 0].astype(float)
            y = data[:, 1].astype(float)
    else:
        raise ValueError(f"Unexpected data shape in {path}: {data.shape}")

    return x, y


def fit_profile(x, y):
    """
    Fit a Gaussian to y(x). Returns:
        popt: best-fit parameters (I0, x0, w, offset)
        perr: 1σ uncertainties on those parameters
    """
    # Background-subtracted copy just for initial guesses
    y_bs = y - np.min(y)

    I0_guess = np.max(y_bs)
    x0_guess = x[np.argmax(y_bs)]
    w_guess  = (x.max() - x.min()) / 10.0
    offset_guess = np.min(y)

    p0 = [I0_guess, x0_guess, w_guess, offset_guess]

    popt, pcov = curve_fit(gaussian_1d, x, y, p0=p0, maxfev=10000)
    perr = np.sqrt(np.diag(pcov))
    return popt, perr


def fit_and_save_plot(path, label, fig_name):
    """
    Load profile, fit Gaussian, save plot, and return
    (I0, x0, w, offset, I0_err, x0_err, w_err, offset_err).
    """
    x, y = load_profile(path)
    popt, perr = fit_profile(x, y)
    I0, x0, w, offset = popt
    I0_err, x0_err, w_err, offset_err = perr

    # Smooth curve for the fit
    x_dense = np.linspace(x.min(), x.max(), 1000)
    y_fit = gaussian_1d(x_dense, *popt)

    # Plot data + fit
    plt.figure()
    plt.plot(x, y, "k.", label="Data")
    plt.plot(x_dense, y_fit, "r-", label="Gaussian fit")
    plt.xlabel("Position (pixel index or given x units)")
    plt.ylabel("Intensity (arb. units)")
    plt.title(f"{label} beam profile with Gaussian fit")
    plt.legend()
    plt.tight_layout()

    # Save plot (no interactive window needed)
    fig_path = os.path.join(output_dir, fig_name)
    plt.savefig(fig_path, dpi=300)
    plt.close()

    print(f"{label}: w = {w:.2f} ± {w_err:.2f} (in x-units, usually pixels)")
    return I0, x0, w, offset, I0_err, x0_err, w_err, offset_err, fig_path


# -------------------------------------------------------
# 3) RUN FITS FOR BEFORE AND AFTER
# -------------------------------------------------------
(I0_b, x0_b, w_before, offset_b,
 I0_b_err, x0_b_err, w_before_err, offset_b_err,
 fig_before) = fit_and_save_plot(before_path, "Before expander",
                                 "BeamExpander_Before.png")

(I0_a, x0_a, w_after, offset_a,
 I0_a_err, x0_a_err, w_after_err, offset_a_err,
 fig_after) = fit_and_save_plot(after_path, "After expander",
                                "BeamExpander_After.png")

# Expansion factor in terms of 1/e^2 radius
M = w_after / w_before

# Propagate relative uncertainties (simple approximation)
rel_err = 0.0
if w_before != 0 and w_after != 0:
    rel_err = np.sqrt((w_before_err / w_before) ** 2 +
                      (w_after_err  / w_after)  ** 2)
M_err = M * rel_err

print("\nMeasured expansion factor (radius):")
print(f"M = w_after / w_before = {M:.2f} ± {M_err:.2f}")
print("Diameter ratio is the same, since diameter = 2 w.")


# -------------------------------------------------------
# 4) APPEND RESULTS TO EXISTING WORD DOCUMENT
# -------------------------------------------------------
doc = Document(output_docx)

# New section for beam expander
doc.add_page_break()
doc.add_heading("Beam Expander Results", level=1)
doc.add_paragraph(
    "This section shows the transverse beam profiles and Gaussian fits "
    "measured before and after the beam expander, along with the "
    "resulting expansion factor."
)

# --- Before expander section ---
doc.add_heading("Before beam expander", level=2)
p_before = doc.add_paragraph()
p_before.add_run(
    f"Gaussian fit parameters (before expander):\n"
    f"I0 = {I0_b:.2f} ± {I0_b_err:.2f}, "
    f"x0 = {x0_b:.2f} ± {x0_b_err:.2f}, "
    f"w = {w_before:.2f} ± {w_before_err:.2f} (pixels, 1/e² radius), "
    f"offset = {offset_b:.2f} ± {offset_b_err:.2f}.\n"
)

if os.path.exists(fig_before):
    doc.add_picture(fig_before, width=Inches(4.5))
else:
    doc.add_paragraph(f"(Figure file not found: {fig_before})")

# --- After expander section ---
doc.add_heading("After beam expander", level=2)
p_after = doc.add_paragraph()
p_after.add_run(
    f"Gaussian fit parameters (after expander):\n"
    f"I0 = {I0_a:.2f} ± {I0_a_err:.2f}, "
    f"x0 = {x0_a:.2f} ± {x0_a_err:.2f}, "
    f"w = {w_after:.2f} ± {w_after_err:.2f} (pixels, 1/e² radius), "
    f"offset = {offset_a:.2f} ± {offset_a_err:.2f}.\n"
)

if os.path.exists(fig_after):
    doc.add_picture(fig_after, width=Inches(4.5))
else:
    doc.add_paragraph(f"(Figure file not found: {fig_after})")

# --- Summary table + expansion factor ---
doc.add_heading("Summary of beam expander performance", level=2)
doc.add_paragraph(
    "The table below lists the fitted 1/e² beam radii before and after the "
    "beam expander. The measured expansion factor is calculated as the ratio "
    "of these waists."
)

table = doc.add_table(rows=1, cols=3)
hdr = table.rows[0].cells
hdr[0].text = "Profile"
hdr[1].text = "w (pixels)"
hdr[2].text = "σ_w (pixels)"

row_b = table.add_row().cells
row_b[0].text = "Before expander"
row_b[1].text = f"{w_before:.2f}"
row_b[2].text = f"{w_before_err:.2f}"

row_a = table.add_row().cells
row_a[0].text = "After expander"
row_a[1].text = f"{w_after:.2f}"
row_a[2].text = f"{w_after_err:.2f}"

doc.add_paragraph(
    f"Measured expansion factor (radius): M = w_after / w_before "
    f"= {M:.2f} ± {M_err:.2f}. The diameter ratio is identical, since "
    "the beam diameter is defined as 2w."
)

# Save updated document
doc.save(output_docx)

print(f"\nUpdated Word document saved as:\n{output_docx}")


Before expander: w = 212.23 ± 3.63 (in x-units, usually pixels)
After expander: w = 557.23 ± 7.95 (in x-units, usually pixels)

Measured expansion factor (radius):
M = w_after / w_before = 2.63 ± 0.06
Diameter ratio is the same, since diameter = 2 w.

Updated Word document saved as:
../results/archived-run/Beam_Diameter_Results.docx
